# Verify two runs give the same `combined_all_bin_averages.npy` — 1-peak

Compares this folder's `simulated_data/shot_averages/combined_all_bin_averages.npy`
(a "new" run) against a reference/previous run's copy of the same file — checking
not just that the swept parameters match, but that the actual noisy data (and
therefore these progressive-heatmap averages) are identical too.

Both files are pickled Python dicts with keys `(bin, n_shots)` and values that
are 2D heatmap arrays (see `extract_heatmaps_subset()` in `get_traindata.ipynb`).

In [1]:
import numpy as np
from pathlib import Path


In [2]:
# --- Set these to the two files you want to compare ---

# This folder's own (new) run
new_path = Path('./simulated_data/shot_averages/combined_all_bin_averages.npy')

# Previous/reference run - EDIT this path to point at another run's output
old_path = Path('/pscratch/sd/x/xchong/3RSE/931/TRS/sim_data/shot_averages_1peak/combined_all_bin_averages.npy')

# Optional companion files (bin -> max shots), compared further below
new_shots_path = new_path.parent / 'shots_per_bin.npy'
old_shots_path = old_path.parent / 'shots_per_bin.npy'

for p in (new_path, old_path):
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")

print(f"New file:  {new_path.resolve()}")
print(f"Old file:  {old_path.resolve()}")


New file:  /pscratch/sd/x/xchong/3RSE/931/simulation/time-resolved-spectroscopy-fit/examples/simulator/simulated_data/shot_averages/combined_all_bin_averages.npy
Old file:  /pscratch/sd/x/xchong/3RSE/931/TRS/sim_data/shot_averages_1peak/combined_all_bin_averages.npy


In [3]:
new_data = np.load(new_path, allow_pickle=True).item()
old_data = np.load(old_path, allow_pickle=True).item()

print(f"New: {len(new_data)} entries")
print(f"Old: {len(old_data)} entries")


New: 2100 entries
Old: 2100 entries


In [4]:
# --- 1. Compare keys: which (bin, n_shots) pairs exist in each ---

new_keys = set(new_data.keys())
old_keys = set(old_data.keys())

common_keys = new_keys & old_keys
only_in_new = new_keys - old_keys
only_in_old = old_keys - new_keys

print(f"Common keys:    {len(common_keys)}")
print(f"Only in new:    {len(only_in_new)}")
print(f"Only in old:    {len(only_in_old)}")

if only_in_new:
    print("\nExample keys only in new run:", sorted(only_in_new)[:5])
if only_in_old:
    print("Example keys only in old run:", sorted(only_in_old)[:5])


Common keys:    2100
Only in new:    0
Only in old:    0


In [5]:
# --- 2. Compare array values for every common key ---
# exact_matches: bit-for-bit identical (np.array_equal)
# close_matches: numerically identical within floating-point tolerance, but not exact
# differing:      meaningfully different

exact_matches = []
close_matches = []
differing = []
max_diffs = {}

for key in sorted(common_keys):
    a = new_data[key]
    b = old_data[key]

    if a.shape != b.shape:
        differing.append(key)
        max_diffs[key] = float('nan')
        continue

    if np.array_equal(a, b):
        exact_matches.append(key)
        max_diffs[key] = 0.0
    else:
        diff = np.abs(a.astype(np.float64) - b.astype(np.float64))
        max_diff = float(diff.max())
        max_diffs[key] = max_diff
        if np.allclose(a, b, rtol=1e-7, atol=1e-10):
            close_matches.append(key)
        else:
            differing.append(key)

print(f"Exact matches (bit-for-bit):     {len(exact_matches)} / {len(common_keys)}")
print(f"Close matches (float tolerance): {len(close_matches)} / {len(common_keys)}")
print(f"Differing:                       {len(differing)} / {len(common_keys)}")


Exact matches (bit-for-bit):     2100 / 2100
Close matches (float tolerance): 0 / 2100
Differing:                       0 / 2100


In [6]:
# --- 3. Summary verdict ---

if not only_in_new and not only_in_old and not differing:
    if close_matches and not exact_matches:
        print("RESULT: Same keys, all values match within floating-point tolerance "
              "(not bit-exact - check dtype/precision if that matters to you).")
    else:
        print("RESULT: IDENTICAL - same keys, all values bit-for-bit identical, including noise.")
else:
    print("RESULT: NOT IDENTICAL.")
    if only_in_new or only_in_old:
        print(f"  - Key sets differ: {len(only_in_new)} only-in-new, {len(only_in_old)} only-in-old")
    if differing:
        worst = sorted(differing, key=lambda k: max_diffs.get(k, 0), reverse=True)[:10]
        print(f"  - {len(differing)} entries differ meaningfully. Worst offenders (largest max abs diff):")
        for k in worst:
            print(f"      {k}: max abs diff = {max_diffs[k]:.6g}")


RESULT: IDENTICAL - same keys, all values bit-for-bit identical, including noise.
